# Production Data Quality Testing

### Validate the production quality-gate module before adding it to Airflow.

In [1]:
# Import sys so the notebook can resolve project modules.
import sys

# Import Path for project-directory handling.
from pathlib import Path


# Detect the repository root.
project_root = Path.cwd().parent


if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Import the production quality runner.
from src.quality import run_core_quality_checks


print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


## Run the quality gate logic

In [2]:
# Run the complete production quality suite.
quality_results = run_core_quality_checks(
    pipeline_name="netflix_incremental_pipeline",
)


# Display every quality result.
for result in quality_results:
    print(
        result["check_name"],
        "→",
        result["status"],
        "| violations:",
        result["observed_value"],
    )


print("PASS: production quality gate completed.")

duplicate_titles → PASS | violations: 0
duplicate_people → PASS | violations: 0
orphan_genre_bridges → PASS | violations: 0
orphan_country_bridges → PASS | violations: 0
orphan_credit_facts → PASS | violations: 0
missing_title_metrics → PASS | violations: 0
PASS: production quality gate completed.


## Confirm the checks were actually audited and not just printed

In [3]:
# Import the shared database helper.
from src.database import get_etl_connection


# Read the latest quality audit records.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT
                check_name,
                status,
                observed_value,
                expected_value,
                checked_at

            FROM etl.quality_results

            WHERE pipeline_name = 'netflix_incremental_pipeline'

            ORDER BY checked_at DESC

            LIMIT 10;
            """
        )

        audit_results = cursor.fetchall()


# Display the persisted quality history.
for row in audit_results:
    print(row)

('missing_title_metrics', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 49, 518077, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
('orphan_credit_facts', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 49, 458571, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
('orphan_country_bridges', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 49, 195414, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
('orphan_genre_bridges', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 49, 113249, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
('duplicate_people', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 48, 958716, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
('duplicate_titles', 'PASS', 0, 0, datetime.datetime(2026, 8, 21, 12, 10, 48, 693869, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos')))
